<a href="https://colab.research.google.com/github/kkeshavsharma/NLP_Practice/blob/main/NLP_Prac.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_parquet("comments_data_13.parquet")

In [ ]:
df.columns

Index(['video_id', 'channel_id', 'comment_id', 'author_display_name',
       'author_channel_id', 'text_display', 'like_count', 'published_at',
       'updated_at'],
      dtype='object')

In [ ]:
df['text_display'] = df['text_display'].str.replace(r'<[^<>]*>', '', regex=True)
df = df.drop_duplicates(subset = ["text_display"])
df= df.groupby('video_id')['text_display'].apply(lambda x: ' '.join(x)).reset_index()


In [ ]:
import pandas as pd
import re

def advanced_nlp_clean(series: pd.Series) -> pd.Series:
    series = series.fillna("").astype(str)

    # 1. Remove URLs and HTML (Your existing fast regex)
    series = series.str.replace(r'https?://\S+|www\.\S+', '', regex=True)
    series = series.str.replace(r'<[^<>]*>', '', regex=True)

    # 2. Normalize possessives (Officer's -> Officer)
    series = series.str.replace(r"['’][sS]\b", '', regex=True)

    # 3. Remove YouTube specific noise (e.g., timestamps like 12:34)
    # Timestamps are common in comments but don't help classify the video type
    series = series.str.replace(r'\b\d{1,2}:\d{2}(:\d{2})?\b', '', regex=True)

    # 4. Handle excessive character repetition (e.g., "goooood" -> "good")
    # This prevents the model from treating "coool" and "cool" as completely unique words
    series = series.str.replace(r'(.)\1{2,}', r'\1\1', regex=True)

    # 5. Clean up extra whitespaces
    series = series.str.replace(r'\s+', ' ', regex=True).str.strip()

    return series

text_df = advanced_nlp_clean(df["text_display"])

In [ ]:
text_df

,text_display
0,CAIIB MAHAPACK 2.0 🔥👉 ENROLL NOW - FOR DISCOUN...
1,same here❤ Cleared in first attemptThankyou ad...
2,"Thank you mam, I cleared exam. Nice class mam ..."
3,please take more sessions. you teach practical...
4,"Thank you very much, Sir"
...,...
3191,JAIIB &amp; CAIIB MAHAPACKS 2.0 - Use Coupon:-...
3192,kis date ki video h ye ?
3193,Thanks a lot ma&#39;am for this wonderful sess...
3194,"Yes audio, video are ok but at the end of the ..."


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize the vectorizer
# max_features=10000 means keep only the top 10k most important words across your dataset
# ngram_range=(1,2) means look at single words ("python") AND two-word phrases ("machine learning")
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

comments = tfidf.fit_transform(comments_per_video['text_display'])


In [ ]:
pd.DataFrame(comments)

,0
0,<Compressed Sparse Row sparse matrix of dtype ...
1,<Compressed Sparse Row sparse matrix of dtype ...
2,<Compressed Sparse Row sparse matrix of dtype ...
3,<Compressed Sparse Row sparse matrix of dtype ...
4,<Compressed Sparse Row sparse matrix of dtype ...
...,...
3189,<Compressed Sparse Row sparse matrix of dtype ...
3190,<Compressed Sparse Row sparse matrix of dtype ...
3191,<Compressed Sparse Row sparse matrix of dtype ...
3192,<Compressed Sparse Row sparse matrix of dtype ...


In [ ]:
!pip install transformers datasets torch

ERROR: Operation cancelled by user


In [ ]:
import torch
from transformers import BertTokenizer, BertModel

# Load pre-trained BERT tokenizer and model (uncased version)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
text = text_df.apply(str).tolist()
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)

# Check token IDs and attention mask
print("Input IDs:", inputs['input_ids'])
print("Attention Mask:", inputs['attention_mask'])


Input IDs: tensor([[  101, 29080, 12322,  ...,     0,     0,     0],
        [  101,  2168,   100,  ...,     0,     0,     0],
        [  101,  4067,  2017,  ...,     0,     0,     0],
        ...,
        [  101,  4283,  1037,  ...,     0,     0,     0],
        [  101,  2748,  5746,  ...,     0,     0,     0],
        [  101,   100,  2961,  ...,     0,     0,     0]])
Attention Mask: tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])


In [ ]:
pd.DataFrame(inputs["input_ids"])

,0,1,2,3,4,5,6,7,8,9,...,502,503,504,505,506,507,508,509,510,511
0,101,29080,12322,24404,23947,1016,1012,1014,100,25612,...,0,0,0,0,0,0,0,0,0,0
1,101,2168,100,5985,1999,2034,3535,21604,4801,7140,...,0,0,0,0,0,0,0,0,0,0
2,101,4067,2017,5003,2213,1010,1045,5985,11360,1012,...,0,0,0,0,0,0,0,0,0,0
3,101,3531,2202,2062,6521,1012,2017,6570,8134,1998,...,0,0,0,0,0,0,0,0,0,0
4,101,4067,2017,2200,2172,1010,2909,102,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3191,101,17410,12322,1004,23713,1025,29080,12322,24404,23947,...,0,0,0,0,0,0,0,0,0,0
3192,101,11382,2015,3058,11382,2678,1044,6300,1029,102,...,0,0,0,0,0,0,0,0,0,0
3193,101,4283,1037,2843,5003,1004,1001,4464,1025,2572,...,0,0,0,0,0,0,0,0,0,0
3194,101,2748,5746,1010,2678,2024,7929,2021,2012,1996,...,0,0,0,0,0,0,0,0,0,0
